# Z-scored sumstats for finemapping input

Make sumstats with columns `SNP`, `CHR`, `BP`, `A1`, `A2`, `Z`.

In [4]:
from pyprojroot.here import here
import polars as pl
import os

In [2]:
processed_dir = here("data/processed/sumstats/")
if not os.path.exists(processed_dir):
  os.makedirs(processed_dir)

## Adams 2025

In [3]:
mdd2025_eur = pl.scan_csv(here("data/raw/sumstats/daner_pgc_mdd_full_eur_hg19_v3.49.24.11.neff.gz"), separator = "\t")

In [16]:
mdd2025_eur_z = (mdd2025_eur
    .select(
        pl.col("SNP"),
        pl.col("CHR"),
        pl.col("BP"),
        pl.col("A1"),
        pl.col("A2"),
        (pl.col("OR").log() / pl.col("SE")).alias("Z")
    )
)

mdd2025_eur_z.sink_csv(here("data/processed/sumstats/mdd2025_eur_hg19_z.tsv"), separator = "\t")
mdd2025_eur_z.sink_parquet(here("data/processed/sumstats/mdd2025_eur_hg19_z.parquet"))

## Meng 2024

Handle rows where `Position` has a float encoding. Remove indels that don't have and ID or specificy alleles.

In [19]:
for cluster in ["AFR", "EAS", "HIS", "SAS"]:
  mdd2024 = pl.scan_csv(
    here(f"data/raw/sumstats/mdd2023diverse_{cluster}_Neff.csv.gz"),
    schema_overrides= {"Position": pl.Float64},
    null_values = ["NA"]
  )

  mdd2024_z = (mdd2024
    .drop_nulls()
    .select(
      pl.col("SNP"),
      pl.col("Chromosome").alias("CHR"),
      pl.col("Position").cast(pl.Int32).alias("BP"),
      pl.col("EA").str.to_uppercase().alias("A1"),
      pl.col("NEA").str.to_uppercase().alias("A2"),
      (pl.col("logOR") / pl.col("SE")).alias("Z")
    )
  )

  mdd2024_z.sink_csv(here(f"data/processed/sumstats/mdd2024_{cluster.lower()}_hg19_z.tsv"), separator = "\t")
  mdd2024_z.sink_parquet(here(f"data/processed/sumstats/mdd2024_{cluster.lower()}_hg19_z.parquet"))